# Step 04 – Model tuning and evaluation of the ten classifiers

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Sections 2.4–2.5 · Table 2, Table 4, Table S4
- **Needs:** step 03 (`RESULTS_DIR/correlation_training_only/`)
- **Writes:** `RESULTS_DIR/train_test_gap/` (tuned models, split indices, test predictions; used by steps 05–10)

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / S2_Features_For_Correlation_v3.csv
#   RESULTS_DIR / correlation_training_only/
#   RESULTS_DIR / train_test_gap/

In [ ]:
!pip install -q catboost xgboost

In [ ]:

import os
print(os.listdir(RESULTS_DIR + 'correlation_training_only/'))

In [ ]:
"""
Training vs. validation vs. test performance for the ten classifiers

Uses the 16 predictors retained by the training-only screening and the SAME
stratified 70/30 split. Re-tunes each model with 5-fold GridSearchCV on the
training partition, using the ORIGINAL hyperparameter grids, and reports:
  Train : resubstitution metrics of the refitted best model (1,960 training samples)
  CV    : mean +/- SD over the 5 validation folds (the model-selection estimate)
  Test  : metrics on the untouched 30% test partition (840 samples)
  Gaps  : Train - Test and CV - Test
Also flags best hyperparameters that fall on the edge of their grid.
Fitted models, best hyperparameters, split indices and test predictions are saved.

In Colab first:  !pip install -q catboost xgboost
"""

import os, time, json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_selection import f_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, make_scorer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# ------------------------------------------------------------------ config
DATA_PATH  = DATA_DIR + 'S2_Features_For_Correlation_v3.csv'
SCREEN_DIR = RESULTS_DIR + 'correlation_training_only/'
OUT_DIR    = RESULTS_DIR + 'train_test_gap/'
LABEL_COL  = 'class'   # <-- set to the label column used in the screening notebook
SEED       = 42        # MUST equal the random_state of the split in the screening notebook
TEST_SIZE  = 0.30
N_FOLDS    = 5
TRAIN_IDX_FILE = None  # set to SCREEN_DIR + 'train_indices.csv' once the screening notebook saves it
SCALER     = StandardScaler  # <-- MinMaxScaler if your original KNN/SVM/MLP pipelines used min-max
RUN_ONLY   = []       # e.g. ['NB'] to re-run some models and merge them into the saved results

PREDICTORS = pd.read_csv(SCREEN_DIR + 'v5_final_vif_training.csv')['feature'].tolist()
assert len(PREDICTORS) == 16, f'Expected 16 predictors, got {len(PREDICTORS)}'

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------------ data and split
df = pd.read_csv(DATA_PATH)
X = df[PREDICTORS]
print('Rows:', len(df), '| class counts:', df[LABEL_COL].value_counts().sort_index().to_dict())
n_nan = X.isna().sum()
assert n_nan.sum() == 0, f'Missing values in predictors:\n{n_nan[n_nan > 0]}'
le = LabelEncoder()
y = pd.Series(le.fit_transform(df[LABEL_COL]), index=df.index)

if TRAIN_IDX_FILE:
    train_idx = pd.read_csv(TRAIN_IDX_FILE).iloc[:, 0].values
    test_idx = df.index.difference(train_idx)
    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)

assert len(X_train) == 1960, 'Training size differs from the screening run - check SEED / TEST_SIZE'

# Proof that this is the same training partition as the screening run:
# the ANOVA F-values must match those in the selection log.
log = pd.read_csv(SCREEN_DIR + 'v5_selection_log_training.csv')
F_now, _ = f_classif(X_train, y_train)
check = pd.DataFrame({'feature': PREDICTORS, 'F_now': F_now}).merge(
    log[['feature', 'F']], on='feature', how='left')
check['diff'] = (check['F_now'] - check['F']).abs()
print(check.round(1).to_string(index=False))
assert check['F'].notna().any(), 'No features matched the selection log - check column names'
assert check['diff'].max() < 0.1, 'Training partition differs from the screening run'

pd.Series(X_train.index, name='train_idx').to_csv(OUT_DIR + 'train_indices.csv', index=False)
pd.Series(X_test.index, name='test_idx').to_csv(OUT_DIR + 'test_indices.csv', index=False)
print('Classes:', dict(zip(le.classes_, range(len(le.classes_)))))
print('Train counts:', np.bincount(y_train), '| Test counts:', np.bincount(y_test))

# ------------------------------------------------------------------ models and ORIGINAL grids
# Tree models: unscaled predictors, no pipeline (grids keep their original names).
# KNN, MDC, MLP, SVM, NB: scaler inside the pipeline, fitted on training folds only.
# NB needs it too: GaussianNB adds var_smoothing * (largest predictor variance) to every
# variance, so one high-variance texture (e.g. prominence) swamps all other predictors.
# Parallel settings mirror the original notebooks so running times stay comparable;
# they do not change the fitted models or the scores.
def pipe(step, est): return Pipeline([('scale', SCALER()), (step, est)])

MODELS = {  # name: (estimator, grid, n_jobs for GridSearchCV)
    'RF': (RandomForestClassifier(random_state=SEED, n_jobs=-1, class_weight='balanced'),
           {'n_estimators': [300, 500, 800],
            'max_depth': [5, 10, 20, 30, 40],
            'min_samples_leaf': [2, 5, 10],
            'max_features': ['sqrt', 0.3, 0.5]}, -1),

    'GTB': (GradientBoostingClassifier(random_state=SEED),
            {'n_estimators': [100, 300],
             'max_depth': [3, 8],
             'min_samples_leaf': [1, 5],
             'learning_rate': [0.01, 0.1],
             'subsample': [0.8, 0.9]}, -1),

    'CatBoost': (CatBoostClassifier(random_seed=SEED, auto_class_weights='Balanced',
                                    verbose=0, thread_count=-1, allow_writing_files=False),
                 {'iterations': [500, 700],
                  'depth': [6, 9],
                  'learning_rate': [0.03, 0.1],
                  'l2_leaf_reg': [1, 3],
                  'rsm': [0.5, 0.9]}, -1),

    'XGBoost': (XGBClassifier(random_state=SEED, eval_metric='mlogloss',
                              tree_method='hist', n_jobs=1),
                {'n_estimators': [400, 600, 800],
                 'max_depth': [6, 10, 15],
                 'learning_rate': [0.01, 0.05, 0.1],
                 'min_child_weight': [1, 3, 5],
                 'colsample_bytree': [0.5, 0.8, 1.0],
                 'subsample': [0.8, 1.0]}, -1),

    'CART': (DecisionTreeClassifier(random_state=SEED),
             {'criterion': ['gini', 'entropy'],
              'max_depth': [5, 10, 15, 20, 25, 30],
              'min_samples_split': [2, 5, 10],
              'min_samples_leaf': [1, 2, 4, 6, 8],
              'max_features': [None, 'sqrt', 'log2']}, -1),

    'NB': (pipe('nb', GaussianNB()),
           {'nb__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]}, -1),

    'KNN': (pipe('knn', KNeighborsClassifier()),
            {'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 21],
             'knn__weights': ['uniform', 'distance'],
             'knn__metric': ['euclidean', 'manhattan']}, -1),

    # 'cosine' removed: scikit-learn >= 1.5 accepts only euclidean/manhattan for NearestCentroid.
    # Scaler added: unscaled, elevation (metres) dominates every distance.
    'MDC': (pipe('mdc', NearestCentroid()),
            {'mdc__metric': ['euclidean', 'manhattan'],
             'mdc__shrink_threshold': [None, 0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0]}, -1),

    'MLP': (pipe('mlp', MLPClassifier(max_iter=1000, random_state=SEED)),  # <-- match original constructor
            {'mlp__hidden_layer_sizes': [(50,), (100,), (100, 50), (50, 25)],
             'mlp__activation': ['relu', 'tanh'],
             'mlp__solver': ['adam'],
             'mlp__alpha': [0.0001, 0.001, 0.01],
             'mlp__learning_rate_init': [0.001, 0.01]}, -1),

    'SVM-RBF': (pipe('svm', SVC()),
                {'svm__C': [1, 3, 5, 10, 20, 50, 100, 200],
                 'svm__gamma': ['scale', 0.001, 0.003, 0.005, 0.01, 0.02, 0.05],
                 'svm__kernel': ['rbf'],
                 'svm__shrinking': [True],
                 'svm__class_weight': ['balanced']}, -1),
}

SCORING = {'OA': 'accuracy', 'kappa': make_scorer(cohen_kappa_score), 'F1': 'f1_macro'}
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def metrics(y_true, y_pred):
    return {'OA': accuracy_score(y_true, y_pred),
            'kappa': cohen_kappa_score(y_true, y_pred),
            'F1': f1_score(y_true, y_pred, average='macro')}

def is_num(v):
    return isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool)

def edge_params(grid, best):
    """Numeric hyperparameters whose best value is the smallest or largest in the grid."""
    flags = []
    for k, vals in grid.items():
        nums = [v for v in vals if is_num(v)]
        if len(nums) >= 2 and is_num(best[k]):
            if best[k] == min(nums):
                flags.append(f'{k}=min({best[k]})')
            elif best[k] == max(nums):
                flags.append(f'{k}=max({best[k]})')
    return '; '.join(flags)

# ------------------------------------------------------------------ fit and evaluate
rows = []
test_preds = pd.DataFrame({'row_idx': X_test.index,
                           'y_true': le.inverse_transform(y_test.values)})
n_test = len(y_test)

for name, (est, grid, n_jobs) in MODELS.items():
    if RUN_ONLY and name not in RUN_ONLY:
        continue
    n_fits = int(np.prod([len(v) for v in grid.values()])) * N_FOLDS
    print(f'\n{name}: {n_fits} fits ...')
    t0 = time.perf_counter()
    gs = GridSearchCV(est, grid, scoring=SCORING, refit='OA', cv=cv,
                      n_jobs=n_jobs, return_train_score=True)
    gs.fit(X_train, y_train)
    gridsearch_time = time.perf_counter() - t0
    best, i, r = gs.best_estimator_, gs.best_index_, gs.cv_results_

    pred_tr = np.ravel(best.predict(X_train)).astype(int)
    t1 = time.perf_counter()
    pred_te = np.ravel(best.predict(X_test)).astype(int)
    predict_time = time.perf_counter() - t1
    tr, te = metrics(y_train, pred_tr), metrics(y_test, pred_te)

    row = {'Model': name}
    for m in ['OA', 'kappa', 'F1']:
        row[f'Train_{m}']         = tr[m]                    # refitted model, full training partition
        row[f'CVfoldTrain_{m}']   = r[f'mean_train_{m}'][i]  # mean over the 5 CV training folds
        row[f'CV_{m}']            = r[f'mean_test_{m}'][i]   # mean over the 5 validation folds
        row[f'CV_{m}_sd']         = r[f'std_test_{m}'][i]
        row[f'Test_{m}']          = te[m]
        row[f'Gap_TrainTest_{m}'] = tr[m] - te[m]
        row[f'Gap_CVTest_{m}']    = r[f'mean_test_{m}'][i] - te[m]
    p = te['OA']
    row['Test_OA_CI95_halfwidth'] = 1.96 * np.sqrt(p * (1 - p) / n_test)  # binomial normal approx.
    row['best_params'] = json.dumps(gs.best_params_, default=str)
    row['edge_params'] = edge_params(grid, gs.best_params_)
    row['n_failed_candidates'] = int(np.isnan(r['mean_test_OA']).sum())
    row['gridsearch_time_s']       = round(gridsearch_time, 2)
    row['refit_time_s']            = round(gs.refit_time_, 3)          # best model, full training set
    row['mean_cv_fold_fit_time_s'] = round(r['mean_fit_time'][i], 3)  # best params, per CV fold
    row['predict_time_s']          = round(predict_time, 4)            # 840 test samples
    rows.append(row)

    joblib.dump(best, OUT_DIR + f'{name}.joblib')
    test_preds[name] = le.inverse_transform(pred_te)
    print(f"{name:9s} train {tr['OA']:.4f} | CV {row['CV_OA']:.4f}+/-{row['CV_OA_sd']:.4f} "
          f"| test {te['OA']:.4f} | gap {tr['OA'] - te['OA']:+.4f} | grid {gridsearch_time/60:.1f} min")
    print('   best:', gs.best_params_)
    if row['edge_params']:
        print('   ON GRID EDGE:', row['edge_params'])

# ------------------------------------------------------------------ save tables
res = pd.DataFrame(rows)
new_params = {r_['Model']: json.loads(r_['best_params']) for r_ in rows}
if RUN_ONLY:   # merge the re-run models into the results saved by the full run
    old = pd.read_csv(OUT_DIR + 'train_cv_test_full.csv')
    res = pd.concat([old[~old['Model'].isin(res['Model'])], res], ignore_index=True)
    old_tp = pd.read_csv(OUT_DIR + 'test_predictions.csv')
    assert (old_tp['row_idx'].values == test_preds['row_idx'].values).all(), 'Test rows differ'
    for m in new_params:
        old_tp[m] = test_preds[m]
    test_preds = old_tp
    with open(OUT_DIR + 'best_params.json') as f:
        new_params = {**json.load(f), **new_params}
res = res.sort_values('Test_OA', ascending=False)
res.to_csv(OUT_DIR + 'train_cv_test_full.csv', index=False)
test_preds.to_csv(OUT_DIR + 'test_predictions.csv', index=False)
with open(OUT_DIR + 'best_params.json', 'w') as f:
    json.dump(new_params, f, indent=2)

pct = lambda v: f'{100 * v:.2f}'
table = pd.DataFrame({
    'Model': res['Model'],
    'Train OA (%)': res['Train_OA'].map(pct),
    'CV OA (%) mean ± SD': [f'{100*a:.2f} ± {100*b:.2f}' for a, b in zip(res['CV_OA'], res['CV_OA_sd'])],
    'Test OA (%) [95% CI]': [f'{100*a:.2f} [±{100*b:.2f}]'
                             for a, b in zip(res['Test_OA'], res['Test_OA_CI95_halfwidth'])],
    'ΔOA Train−Test (pp)': (100 * res['Gap_TrainTest_OA']).map(lambda v: f'{v:+.2f}'),
    'ΔOA CV−Test (pp)': (100 * res['Gap_CVTest_OA']).map(lambda v: f'{v:+.2f}'),
    'Train κ': res['Train_kappa'].map(lambda v: f'{v:.4f}'),
    'Test κ': res['Test_kappa'].map(lambda v: f'{v:.4f}'),
    'Train F1': res['Train_F1'].map(lambda v: f'{v:.4f}'),
    'Test F1': res['Test_F1'].map(lambda v: f'{v:.4f}'),
})
table.to_csv(OUT_DIR + 'table_train_vs_test.csv', index=False)
print('\n', table.to_string(index=False))
print('\nEdge flags:\n', res[['Model', 'edge_params', 'n_failed_candidates']].to_string(index=False))
res[['Model', 'gridsearch_time_s', 'refit_time_s', 'mean_cv_fold_fit_time_s', 'predict_time_s']].to_csv(
    OUT_DIR + 'running_times.csv', index=False)
print('\nFiles saved to:', OUT_DIR)

In [ ]:
# Optional diagnostic: why GaussianNB is scaled (one high-variance predictor dominates var_smoothing)
v = X_train.var().sort_values()
print(v.tail(3)); print('epsilon added:', 1e-9 * v.max(), '| smallest variances:', v.head(3).round(5).to_dict())

## Table 2 hyperparameters and computing environment
Run in the same session as the model runs above, so the recorded hardware matches running_times.csv.

In [ ]:
"""
Table 2 hyperparameter names and computing environment

Run this as the LAST cell of this notebook and run it in the SAME Colab session,
right after the models finish. Colab can assign a different machine after a restart,
so the hardware recorded here must be the one that produced running_times.csv.
It needs the MODELS dictionary and OUT_DIR from that notebook (if the session was
restarted, re-run the cells that define them - no refitting needed - but then the
hardware may no longer match the runtimes).

Outputs (OUT_DIR):
  environment.json               CPU, cores, GPU, RAM, OS, Python and library versions
  table2_hyperparameters.csv     Table 2 built from the objects actually used:
                                 exact parameter names, search values, selected value,
                                 library + version, and non-default fixed settings
"""

import os, sys, json, platform, subprocess
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
import pandas as pd
import psutil

# ------------------------------------------------------------------ computing environment
def cpu_model():
    try:
        with open('/proc/cpuinfo') as f:
            for line in f:
                if line.startswith('model name'):
                    return line.split(':', 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or 'unknown'

def gpu_info():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=10)
        return out.stdout.strip() or 'None'
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return 'None'

def os_name():
    try:
        with open('/etc/os-release') as f:
            for line in f:
                if line.startswith('PRETTY_NAME'):
                    return line.split('=', 1)[1].strip().strip('"')
    except OSError:
        pass
    return platform.platform()

def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return 'not installed'

PACKAGES = ['numpy', 'pandas', 'scikit-learn', 'xgboost', 'catboost',
            'esda', 'libpysal', 'geopandas', 'matplotlib']
env = {
    'platform': 'Google Colab' if 'google.colab' in sys.modules else 'local',
    'cpu_model': cpu_model(),
    'logical_cpus': os.cpu_count(),
    'physical_cores': psutil.cpu_count(logical=False),
    'gpu': gpu_info(),
    'ram_GiB': round(psutil.virtual_memory().total / 1024**3, 1),
    'os': os_name(),
    'python': platform.python_version(),
    'packages': {p: pkg_version(p) for p in PACKAGES},
    'recorded_at': datetime.now().strftime('%Y-%m-%d %H:%M'),
}
# environment.json = latest session; the time-stamped copy keeps every session on record
stamp = datetime.now().strftime('%Y%m%d_%H%M')
for fname in ['environment.json', f'environment_{stamp}.json']:
    with open(OUT_DIR + fname, 'w') as f:
        json.dump(env, f, indent=2)
print(json.dumps(env, indent=2))

# ------------------------------------------------------------------ Table 2 hyperparameters
LIBS = {'sklearn': 'scikit-learn', 'xgboost': 'xgboost', 'catboost': 'catboost'}
IGNORE = {'random_state', 'random_seed', 'n_jobs', 'thread_count', 'verbose',
          'allow_writing_files'}          # reproducibility/runtime settings, not hyperparameters
best = json.load(open(OUT_DIR + 'best_params.json'))

def fmt(v):
    if isinstance(v, list):          # JSON turns tuples such as (100, 50) into lists
        v = tuple(v)
    return str(v).replace("'", '')

def non_default(est):
    """Constructor settings that differ from the library default (e.g. class weights)."""
    default = type(est)().get_params()
    return {k: v for k, v in est.get_params().items()
            if k not in IGNORE and repr(default.get(k, '__unset__')) != repr(v)}

rows = []
for name, (est, grid, _) in MODELS.items():
    final = est.steps[-1][1] if hasattr(est, 'steps') else est
    lib = LIBS.get(type(final).__module__.split('.')[0], type(final).__module__)
    lib_v = f'{lib} {pkg_version(lib)}'
    cls = type(final).__name__
    for k, vals in grid.items():
        pname = k.split('__')[-1]                       # drop pipeline prefix (knn__, svm__ ...)
        rows.append({'Model': name, 'Library': lib_v, 'Class': cls,
                     'Hyperparameter': pname,
                     'Search values': ', '.join(fmt(v) for v in vals),
                     'Selected': fmt(best[name][k])})
    fixed = non_default(final)
    grid_names = {k.split('__')[-1] for k in grid}
    for k, v in fixed.items():
        if k not in grid_names:
            rows.append({'Model': name, 'Library': lib_v, 'Class': cls,
                         'Hyperparameter': k, 'Search values': '(fixed)', 'Selected': fmt(v)})
    if hasattr(est, 'steps'):
        rows.append({'Model': name, 'Library': lib_v, 'Class': cls,
                     'Hyperparameter': '(preprocessing)', 'Search values': '(fixed)',
                     'Selected': type(est.steps[0][1]).__name__})

table2 = pd.DataFrame(rows)

# Two-stage search: if the stage-1 snapshot exists, show both stages side by side.
STAGE1_TABLE = OUT_DIR.rstrip('/') + '_stage1/table2_hyperparameters.csv'
if os.path.exists(STAGE1_TABLE):
    s1 = pd.read_csv(STAGE1_TABLE)[['Model', 'Hyperparameter', 'Search values', 'Selected']]
    s1 = s1.rename(columns={'Search values': 'Search values (stage 1)',
                            'Selected': 'Selected (stage 1)'})
    table2 = table2.rename(columns={'Search values': 'Search values (stage 2)',
                                    'Selected': 'Selected (final)'})
    table2 = table2.merge(s1, on=['Model', 'Hyperparameter'], how='left')
    same = table2['Search values (stage 2)'] == table2['Search values (stage 1)']
    table2.loc[same, 'Search values (stage 2)'] = '-'          # model not refined
    table2 = table2[['Model', 'Library', 'Class', 'Hyperparameter',
                     'Search values (stage 1)', 'Selected (stage 1)',
                     'Search values (stage 2)', 'Selected (final)']]
    print('Stage-1 snapshot found: Table 2 shows both search stages.')

table2.to_csv(OUT_DIR + 'table2_hyperparameters.csv', index=False)
pd.set_option('display.max_colwidth', 60)
print(table2.to_string(index=False))